In [ ]:
from polsar_online_rf import OnlinePolSARRandomForest, NodeTestSampler
import sys
import os
import rasterio as rs
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from sklearn.model_selection import train_test_split

In [ ]:
for i in range(1,351):
    tiff_HH = rs.open('../test/train_set/AIR-PolarSAR-Seg-'+str(i)+'_HH.tiff')
    tiff_HV = rs.open('../test/train_set/AIR-PolarSAR-Seg-'+str(i)+'_HV.tiff')
    tiff_VH = rs.open('../test/train_set/AIR-PolarSAR-Seg-'+str(i)+'_VH.tiff')
    tiff_VV = rs.open('../test/train_set/AIR-PolarSAR-Seg-'+str(i)+'_VV.tiff')
    HH_data = tiff_HH.read().flatten().reshape((512,512))
    HV_data = tiff_HV.read().flatten().reshape((512,512))
    VH_data = tiff_VH.read().flatten().reshape((512,512))
    VV_data = tiff_VV.read().flatten().reshape((512,512))
    new_dataset = rs.open(
    '../test/train_set/AIR-PolarSAR-Seg-'+str(i)+'_Combined.tiff',
    'w',
    driver='GTiff',
    height=tiff_HH.shape[0],
    width=tiff_HH.shape[1],
    count=4,
    dtype=tiff_HH.dtypes[0],
    crs='+proj=latlong',)
    new_dataset.write(HH_data,1)
    new_dataset.write(HV_data,2)
    new_dataset.write(VH_data,3)
    new_dataset.write(VV_data,4)
    new_dataset.close()

In [ ]:
from skimage import io

igbp2hunan = np.array([255, 0, 1, 2, 1, 3, 4, 6, 6, 5, 6, 7, 255])

def load_lc(path):
    lc = io.imread(path)
    lc[lc == 255] = 12
    lc = igbp2hunan[lc]
    return lc

In [ ]:
cleaned_lc = load_lc('../../Hunan_Dataset/Hunan_Dataset/train/lc/lc_299.tif')

In [ ]:
plt.imshow(cleaned_lc)

In [ ]:
s1 = rs.open('../../Hunan_Dataset/Hunan_Dataset/train/s1/s1_299.tif')

In [ ]:
s1.count

In [ ]:
plt.imshow(np.array(s1.read(2)))

In [ ]:
dataset = rs.open('../../Hunan_Dataset/Hunan_Dataset/train/tri/tri_299.tif')

In [ ]:
dataset.read(1)

In [ ]:
plt.imshow(np.array(dataset.read(1)))

In [ ]:
file_list_aid = os.listdir('../../UAE-RS/AID')
file_list_ucm = os.listdir('../../UAE-RS/UCM')
aid_cat = pl.Series(map(lambda x: x.split('.')[0].split('_')[0],file_list_aid))
ucm_cat = pl.Series(map(lambda x: x.split('.')[0].split('_')[0][0:-2],file_list_ucm))
aid_df = pl.DataFrame({"file_name":file_list_aid,"cat":aid_cat})
ucm_df = pl.DataFrame({"file_name":file_list_ucm,"cat":ucm_cat})

In [ ]:
intersection = aid_cat.filter(aid_cat.is_in(ucm_cat)).unique()
intersection

In [ ]:
X = aid_df.filter(aid_df["cat"].is_in(intersection))["file_name"]
Y = aid_df.filter(aid_df["cat"].is_in(intersection))["cat"]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.3)

In [ ]:
NodeTestSampler()